In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc_context
import matplotlib.patheffects as path_effects
from matplotlib.collections import LineCollection
from matplotlib import patches
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Polygon
from matplotlib import animation
import sunpy
import sunpy.map
from sunpy.coordinates import propagate_with_solar_surface, get_earth
import astropy
from astropy.coordinates import SkyCoord
import astropy.units as u
import astropy.constants as const
from astropy.io import fits, ascii
from astropy.time import Time
from astropy.convolution import convolve, Gaussian2DKernel
from astropy.wcs import WCS
from astropy.visualization import ImageNormalize, AsinhStretch
from streamtracer import StreamTracer, VectorGrid
from extrapolater import PotentialField
from helpers import from_local
from scipy.spatial import cKDTree
from tqdm.notebook import tqdm

import h5py 
import dask.array as da 
from ndcube import NDCube
from ndcube.wcs.tools import unwrap_wcs_to_fitswcs
from fancy_colorbar import plot_colorbar, wcs_scalebar
from IPython.display import HTML, display

import mhsxtrapy

In [23]:
ms_style_dict = {'text.usetex': True, 'font.family': 'serif',
                 'font.serif': ['Computer Modern Roman'], 'axes.linewidth': 1.2,
                 'xtick.major.width': 1.2, 'xtick.major.size': 4,
                 'ytick.major.width': 1.2, 'ytick.major.size': 4,
                 'xtick.minor.width': 1.2, 'xtick.minor.size': 2,
                 'ytick.minor.width': 1.2, 'ytick.minor.size': 2,
                 'xtick.direction': 'in', 'ytick.direction': 'in',
                 'text.latex.preamble': r'\usepackage[T1]{fontenc}'
                 r'\usepackage{amsmath}' r'\usepackage{siunitx}'
                 r'\sisetup{detect-all=True}' r'\usepackage{fixltx2e}'}

In [3]:
def colored_line_with_alpha(x, y, c, ax, norm_color, norm_alpha=None,
    alpha=None, **lc_kwargs):
    """
    Plot a line with a color specified along the line by a third value.

    It does this by creating a collection of line segments. Each line segment is
    made up of two straight lines each connecting the current (x, y) point to the
    midpoints of the lines connecting the current point with its two neighbors.
    This creates a smooth line with no gaps between the line segments.

    Parameters
    ----------
    x, y : array-like
        The horizontal and vertical coordinates of the data points.
    c : array-like
        The color values, which should be the same size as x and y.
    ax : Axes
        Axis object on which to plot the colored line.
    **lc_kwargs
        Any additional arguments to pass to matplotlib.collections.LineCollection
        constructor. This should not include the array keyword argument because
        that is set to the color argument. If provided, it will be overridden.

    Returns
    -------
    matplotlib.collections.LineCollection
        The generated line collection representing the colored line.
    """
    if "array" in lc_kwargs:
        warnings.warn('The provided "array" keyword argument will be overridden')

    # Default the capstyle to butt so that the line segments smoothly line up
    default_kwargs = {"capstyle": "butt", "rasterized": True}
    default_kwargs.update(lc_kwargs)

    # Compute the midpoints of the line segments. Include the first and last points
    # twice so we don't need any special syntax later to handle them.
    x = np.asarray(x)
    y = np.asarray(y)
    x_midpts = np.hstack((x[0], 0.5 * (x[1:] + x[:-1]), x[-1]))
    y_midpts = np.hstack((y[0], 0.5 * (y[1:] + y[:-1]), y[-1]))

    # Determine the start, middle, and end coordinate pair of each line segment.
    # Use the reshape to add an extra dimension so each pair of points is in its
    # own list. Then concatenate them to create:
    # [
    #   [(x1_start, y1_start), (x1_mid, y1_mid), (x1_end, y1_end)],
    #   [(x2_start, y2_start), (x2_mid, y2_mid), (x2_end, y2_end)],
    #   ...
    # ]
    coord_start = np.column_stack((x_midpts[:-1], y_midpts[:-1]))[:, np.newaxis, :]
    coord_mid = np.column_stack((x, y))[:, np.newaxis, :]
    coord_end = np.column_stack((x_midpts[1:], y_midpts[1:]))[:, np.newaxis, :]
    segments = np.concatenate((coord_start, coord_mid, coord_end), axis=1)

    cmap = plt.get_cmap(default_kwargs.pop("cmap", "GnBu_r"))
    color_rgb = cmap(norm_color(c))

    if alpha is not None:
        color_rgb[:,-1] = alpha
    else:
        color_rgb[:,-1] = 1 - norm_alpha(c)*0.9

    lc = LineCollection(segments, colors=color_rgb, **default_kwargs)
    # lc.set_array(c)  # set the colors of each segment

    return ax.add_collection(lc)

In [4]:
file_Hbeta_pr = h5py.File("/cluster/home/zhuyin/work/dkist_solo_fibril_data/pid_1_123_aux/plot_ready/Hbeta_BJOLO_pr.hdf5")
Hbeta_pr_set = file_Hbeta_pr["vbi_img"]
Hbeta_pr_da = da.from_array(Hbeta_pr_set, chunks=(1, 4096 - 128*2, 4096 - 128*2))
Hbeta_date_obs = Time(ascii.read("/cluster/home/zhuyin/work/dkist_solo_fibril_data/pid_1_123_aux/plot_ready/Hbeta_BJOLO_date_avg.txt")["DATE-AVG"])

In [5]:
# file_Halpha_pr = h5py.File("/cluster/home/zhuyin/work/dkist_solo_fibril_data/pid_1_123_aux/plot_ready/Halpha_BLZNL_pr.hdf5")
# Halpha_pr_set = file_Halpha_pr["vbi_img"]
# Halpha_pr_da = da.from_array(Halpha_pr_set, chunks=(1, 4096 - 128*2, 4096 - 128*2))
Halpha_date_obs = Time(ascii.read("/cluster/home/zhuyin/work/dkist_solo_fibril_data/pid_1_123_aux/plot_ready/Halpha_BLZNL_date_avg.txt")["DATE-AVG"])

In [6]:
file_hri_pr_noproj_dset = h5py.File("/cluster/home/zhuyin/work/dkist_solo_fibril_data/pid_1_123_aux/plot_ready/HRIEUV_noproj_pr.hdf5")
hri_pr_noproj_array = file_hri_pr_noproj_dset["hrieuv_noproj_img"][:]
hrieuv_no_proj_extent = np.load("/cluster/home/zhuyin/work/dkist_solo_fibril_data/pid_1_123_aux/plot_ready/hrieuv_no_proj_extent.npy")
hrieuv_nocrop_noproj_wcs = WCS(fits.getheader("/cluster/home/zhuyin/work/dkist_solo_fibril_data/pid_1_123_aux/plot_ready/hri_nocrop_noproj_wcs.fits",
                                        ignore_missing_simple=True))
hrieuv_noproj_wcs = hrieuv_nocrop_noproj_wcs[hrieuv_no_proj_extent[1]:hrieuv_no_proj_extent[3] + 1,
                                hrieuv_no_proj_extent[0]:hrieuv_no_proj_extent[2] + 1]
hrieuv_date_ear = Time(ascii.read("/cluster/home/zhuyin/work/dkist_solo_fibril_data/pid_1_123_aux/plot_ready/HRIEUV_date_ear.txt")["date_ear"])

In [7]:
dkist_vbi_target_header = fits.getheader("../../data/pid_1_123_aux/plot_ready/dkist_target_wcs_header_before_crop.fits",
                                        ignore_missing_simple=True)

dkist_vbi_target_data = np.zeros((4096,4096))

dkist_vbi_target_cube = NDCube(dkist_vbi_target_data,WCS(dkist_vbi_target_header, naxis=2))
dkist_vbi_target_cube_crop = dkist_vbi_target_cube[128:-128,128:-128]
dkist_vbi_target_cube_crop_rebin = dkist_vbi_target_cube_crop.rebin((8,8))

Set MJD-BEG to 59876.791095 from DATE-BEG.
Set MJD-AVG to 59876.791095 from DATE-AVG.
Set MJD-END to 59876.791095 from DATE-END'. [astropy.wcs.wcs]
Set OBSGEO-B to    20.706700 from OBSGEO-[XYZ].
Set OBSGEO-H to     3063.997 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


In [8]:
data3d = mhsxtrapy.ExtrapolationResult.load("../../data/pid_1_123_aux/MHSXtra_results_new/alpha_0.0120/field3d.h5")

nx, ny, nz, nf = 960, 536, 536, 536
# xmin, xmax, ymin, ymax, zmin, zmax = 0.0, 1.0, 0.0, 1.0, 0.0, 1.0

pixelsize_x = 0.23712652199468398
pixelsize_y = pixelsize_x
pixelsize_z = 0.05

# x_arr = np.linspace(xmin, xmax, nx, dtype=np.float64)
# y_arr = np.linspace(ymin, ymax, ny, dtype=np.float64)
# z_arr = np.linspace(zmin, zmax, nz, dtype=np.float64)
x_arr = np.arange(nx) * pixelsize_x
y_arr = np.arange(ny) * pixelsize_y
z_arr = np.arange(nz) * pixelsize_z

bx_extra = data3d.field[:,:,:,1]
by_extra = data3d.field[:,:,:,0]
bz_extra = data3d.field[:,:,:,2]

bx_extra = bx_extra[ny:ny*2, nx:nx*2,:]
by_extra = by_extra[ny:ny*2, nx:nx*2,:]
bz_extra = bz_extra[ny:ny*2, nx:nx*2,:]

field_array = np.array([bx_extra.transpose(1,0,2), by_extra.transpose(1,0,2), bz_extra.transpose(1,0,2)]).transpose(1,2,3,0)

field_grid = VectorGrid(field_array, grid_coords=[x_arr, y_arr, z_arr])

In [9]:
bz_extra.transpose(1,0,2).shape

(960, 536, 536)

In [10]:
x_select = x_arr[420:590]
y_select = y_arr[220:380]
mesh_X, mesh_Y = np.meshgrid(x_select, y_select)
pts = np.column_stack((mesh_X.ravel(), mesh_Y.ravel()))
weights = np.sqrt(np.abs(bz_extra[220:380,420:590,0].T)).ravel()
weights = weights/np.sum(weights) + 1e-30

In [11]:
def flux_weighted_seeds(x_select=x_select, y_select=y_select, 
            n_seeds=200, min_sep=None, weights=weights, z0=z_arr.min(),
            gap_factor=1.5):
    idx = np.random.choice(pts.shape[0], size=5*n_seeds, replace=True, p=weights)
    cand = pts[idx]
    # Poisson-like thinning using a distance threshold
    chosen = []
    if min_sep is None:
        # ~1–2 grid pixels is a good start
        min_sep = gap_factor*min(np.diff(x_select).mean(), np.diff(y_select).mean())
    tree = None
    for p in cand:
        if not chosen:
            chosen.append(p)
            tree = cKDTree(np.array(chosen))
        else:
            if tree.query(p, k=1)[0] >= min_sep:
                chosen.append(p); tree = cKDTree(np.array(chosen))
                if len(chosen) >= n_seeds: break
    S = np.array(chosen)
    return np.column_stack([S, np.full(len(S), z0)])

In [12]:
# seeds_test = flux_weighted_seeds(n_seeds=500, gap_factor=5)
# np.save("../../sav/sotsp_seeds.npy", seeds_test)

In [13]:
seeds_test = np.load("../../sav/sotsp_seeds.npy")

In [14]:
nsteps = 10000
step_size = 0.02
tracer = StreamTracer(nsteps, step_size)
tracer.trace(seeds_test, field_grid)

In [15]:
sotsp_br_map = sunpy.map.Map("../../data/pid_1_123_aux/SOTSP/sotsp_bz_hgs_cea_fullfov.fits")

fline_hgs = []

for fline in tracer.xs[:]:
    # if fline[-1,2] < 60:
    fline_hgs_ = sotsp_br_map.wcs.pixel_to_world(fline[:,0]/pixelsize_x, fline[:,1]/pixelsize_y)
    fline_hgs_ = SkyCoord(lon=fline_hgs_.lon, lat=fline_hgs_.lat, radius=fline[:,2]*u.Mm + 695700*u.km,
                        frame=fline_hgs_.frame)
    
    fline_hgs.append(fline_hgs_)

fline_vbi_pixel = []

for fline in fline_hgs[:]:
    with propagate_with_solar_surface():
        fline_pixel_x, fline_pixel_y = dkist_vbi_target_cube_crop.wcs.world_to_pixel(fline)
    fline_vbi_pixel.append([fline_pixel_x, fline_pixel_y, fline.radius.to_value(u.Mm) - 695.7])

fline_hri_pixel = []
for fline in fline_hgs[:]:
    with propagate_with_solar_surface():
        fline_pixel_x, fline_pixel_y = hrieuv_noproj_wcs.world_to_pixel(fline)
    fline_hri_pixel.append([fline_pixel_x, fline_pixel_y, fline.radius.to_value(u.Mm) - 695.7])

In [16]:
del bx_extra, by_extra, bz_extra, field_array

In [17]:
def fieldline_radial_angle_lat(r, lat, lon, input_degrees=True, output_degrees=True):
    import numpy as np

    r = np.asarray(r)
    lat = np.asarray(lat)
    lon = np.asarray(lon)

    if input_degrees:
        lat = np.deg2rad(lat)
        lon = np.deg2rad(lon)

    lon = np.unwrap(lon)

    dr = np.gradient(r)
    dlat = np.gradient(lat)
    dlon = np.gradient(lon)

    tangent_norm = np.sqrt(
        dr**2 +
        (r * dlat)**2 +
        (r * np.cos(lat) * dlon)**2
    )

    cos_alpha = dr / tangent_norm
    cos_alpha = np.clip(cos_alpha, -1.0, 1.0)

    alpha = np.arccos(cos_alpha)

    if output_degrees:
        alpha = np.rad2deg(alpha)

    return alpha

In [18]:
fieldline_radial_angle_list = []

for fline in fline_hgs:
    fieldline_radial_angles = fieldline_radial_angle_lat(fline.radius.to_value(u.Mm),
        fline.lat.to_value(u.deg),
        fline.lon.to_value(u.deg))
    
    fieldline_radial_angles = np.minimum(fieldline_radial_angles, 180 - fieldline_radial_angles)
    fieldline_radial_angle_list.append(fieldline_radial_angles)

In [19]:
w_space = 0.06
h_space = 0.05
left_margin = 0.23
right_margin = 0.02
top_margin = 0.24
bottom_margin = 0.13
fig_size_scale = 2.7
fig_x = (left_margin + 2 + w_space + right_margin) 
fig_y = (top_margin + 2 + h_space + bottom_margin)

In [20]:
fig_x*fig_size_scale, fig_y*fig_size_scale

(6.237000000000001, 6.534)

In [21]:
def make_movie(save_dir=None, fps=10, dpi=150):
    with plt.rc_context(ms_style_dict):
        plt.close("all")

        hrieuv_index_ = 0
        Hbeta_index_ = np.nanargmin(np.abs(hrieuv_date_ear[hrieuv_index_] - Hbeta_date_obs))

        fig = plt.figure(figsize=(fig_x*fig_size_scale, fig_y*fig_size_scale))

        ax1 = fig.add_axes([left_margin/fig_x, (bottom_margin + 1 + h_space)/fig_y,
                1/fig_x, 1/fig_y],
                projection=unwrap_wcs_to_fitswcs(dkist_vbi_target_cube_crop.wcs)[0])

        im1 = ax1.imshow(Hbeta_pr_da[Hbeta_index_,:,:], origin="lower", cmap="Greys_r",
                norm=ImageNormalize(vmin=-0.2,vmax=1.1),
                interpolation="none",
                rasterized=True)

        roiA_verts = np.array([[50,50], [2048,50],
            [2048, 1536], [1536, 2560],
            [50, 2560], [50, 50]])
        
        ax1.text(50+64, 2560 - 64,
            r"\textbf{A}", ha="left", va="top", fontsize=11, color="#E83015",
            alpha=1, path_effects=[path_effects.Stroke(linewidth=2, foreground="w"),
            path_effects.Normal()]
            )

        roiB_verts = np.array([[1536+64, 2560], [3072, 2560],
            [3072, 1200],
            [1536+64 - (2560 - 1200)*(1536 - 2048)/(2560 - 1536),1200]])
        
        ax1.text(3072 - 64, 2560 - 64,
            r"\textbf{B}", ha="right", va="top", fontsize=11, color="#E83015",
            alpha=1, path_effects=[path_effects.Stroke(linewidth=2, foreground="w"),
            path_effects.Normal()]
            )

        roiC_verts = np.array([[832, 2560+64], [3608, 2560+64],
            [3608, 3840 - 50], [832, 3840 - 50]])
        
        ax1.text(832 + 64, 3840 - 50 - 64,
            r"\textbf{C}", ha="left", va="top", fontsize=11, color="#E83015",
            alpha=1, path_effects=[path_effects.Stroke(linewidth=2, foreground="w"),
            path_effects.Normal()]
            )

        for verts_ in (roiA_verts, roiB_verts, roiC_verts):
            poly_ = Polygon(
                verts_, closed=True,
                facecolor='none', 
                edgecolor='#E83015', lw=2,
                ls=(0,(5,3)), alpha=0.9,
                joinstyle='round',
            )

            ax1.add_patch(poly_)

        sb1 = wcs_scalebar(ax1, 2*u.Mm, corner="bottom left",
                    label=r"\textbf{2 Mm}",
                    dsun=get_earth(Hbeta_date_obs[Hbeta_index_]).radius,
                    bbox_to_anchor=(0.81, 0.02, 0.15, 0.15),
                    bbox_transform=ax1.transAxes,
                    frame=False, 
                    color="white", size_vertical=35,
                    fontproperties={"size": 11},
                    sep=8,
                    pad=0,
                    borderpad=0
                    )
        
        fig.canvas.draw()
        
        sb1.txt_label._text.set_path_effects([
            path_effects.Stroke(linewidth=2, foreground='black'),
            path_effects.Normal()
        ])

        sb1.size_bar.get_children()[0].set_path_effects([
            path_effects.Stroke(linewidth=2, foreground='black'),
            path_effects.Normal()
        ])

        ax2 = fig.add_axes([(left_margin + 1 + w_space)/fig_x, (bottom_margin + 1 + h_space)/fig_y,
                1/fig_x, 1/fig_y],
                projection=unwrap_wcs_to_fitswcs(dkist_vbi_target_cube_crop.wcs)[0])

        im2 = ax2.imshow(Hbeta_pr_da[Hbeta_index_,:,:], origin="lower", cmap="Greys_r",
                norm=ImageNormalize(vmin=-0.2,vmax=1.1),
                interpolation="none",
                rasterized=True)
        
        ax3 = fig.add_axes([(left_margin)/fig_x, bottom_margin/fig_y,
                1/fig_x, 1/fig_y], projection=hrieuv_noproj_wcs)
        
        im3 = ax3.imshow(hri_pr_noproj_array[hrieuv_index_],
                    cmap="solar orbiterhri_euv174",
                    interpolation="none", aspect=1,
                    rasterized=True)

        # ax3.set_anchor((0,0))

        sb3 = wcs_scalebar(ax3, 2*u.Mm, corner="bottom left",
                    label=r"\textbf{2 Mm}",
                    # dsun=get_earth(Hbeta_date_obs[Hbeta_index_]).radius,
                    bbox_to_anchor=(0.81, 0.02, 0.15, 0.15),
                    bbox_transform=ax3.transAxes,
                    frame=False, 
                    color="white",
                    size_vertical=35*hri_pr_noproj_array[hrieuv_index_].shape[1]/\
                        Hbeta_pr_set[Hbeta_index_].shape[1],
                    fontproperties={"size": 11},
                    sep=8,
                    pad=0,
                    borderpad=0
                    )
        
        fig.canvas.draw()
        
        sb3.txt_label._text.set_path_effects([
            path_effects.Stroke(linewidth=2, foreground='black'),
            path_effects.Normal()
        ])

        sb3.size_bar.get_children()[0].set_path_effects([
            path_effects.Stroke(linewidth=2, foreground='black'),
            path_effects.Normal()
        ])

        ax4 = fig.add_axes([(left_margin + 1 + w_space)/fig_x, bottom_margin/fig_y,
                1/fig_x, 1/fig_y], projection=hrieuv_noproj_wcs)
        
        im4 = ax4.imshow(hri_pr_noproj_array[hrieuv_index_],
                    cmap="solar orbiterhri_euv174",
                    interpolation="none", aspect=1,
                    rasterized=True)
        
        for ax_ in (ax1,ax2,ax3,ax4):
            ax_.autoscale(False)
        
        for fline in fline_vbi_pixel[:]:
            colored_line_with_alpha(fline[0], fline[1], fline[2], ax1, cmap="summer",
            norm_color=ImageNormalize(vmin=0,vmax=2, clip=True),
            norm_alpha=ImageNormalize(vmin=0,vmax=5, clip=True),
            lw=1.5)

        for fline, fline_radial_angle in zip(fline_vbi_pixel, fieldline_radial_angle_list):
            colored_line_with_alpha(fline[0], fline[1], fline_radial_angle, ax2, cmap="GnBu_r",
            norm_color=ImageNormalize(vmin=0,vmax=90, clip=True),
            alpha=1 - ImageNormalize(vmin=0,vmax=10, clip=True)(fline[2])*0.9,
            lw=1.5)

        for fline in fline_hri_pixel[:]:
            colored_line_with_alpha(fline[0], fline[1], fline[2], ax3, cmap="summer",
            norm_color=ImageNormalize(vmin=0,vmax=2, clip=True),
            norm_alpha=ImageNormalize(vmin=0,vmax=5, clip=True),
            lw=1.5)

        for fline, fline_radial_angle in zip(fline_hri_pixel, fieldline_radial_angle_list):
            colored_line_with_alpha(fline[0], fline[1], fline_radial_angle, ax4, cmap="GnBu_r",
            norm_color=ImageNormalize(vmin=0,vmax=90, clip=True),
            alpha=1 - ImageNormalize(vmin=0,vmax=10, clip=True)(fline[2])*0.9,
            lw=1.5)


        line_colormappable = ScalarMappable(norm=ImageNormalize(vmin=0,vmax=2, clip=True), cmap="summer")
        
        clb, clb_ax = plot_colorbar(line_colormappable, ax1, bbox_to_anchor=(0.1, 1.03, 0.8, 0.06),
            orientation="horizontal", extend="max")
        clb_ax.set_xlabel("Altitude (Mm)", fontsize=11)

        clb_ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False, which="both")
        clb_ax.xaxis.set_label_position('top')

        line_colormappable_inc = ScalarMappable(norm=ImageNormalize(vmin=0,vmax=90, clip=True), cmap="GnBu_r")

        clb_inc, clb_inc_ax = plot_colorbar(line_colormappable_inc, ax2, bbox_to_anchor=(0.1, 1.03, 0.8, 0.06),
            orientation="horizontal", extend="max")
        clb_inc_ax.set_xlabel("Inclination (deg)", fontsize=11)
        clb_inc_ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False, which="both")
        clb_inc_ax.xaxis.set_label_position('top')
        clb_inc_ax.set_xticks([0, 30, 60, 90])

        ax1.coords[0].set_axislabel("", visible=False)
        ax1.coords[1].set_axislabel("", visible=False)

        for ax_ in (ax2,ax4):
            ax_.coords[0].set_ticklabel_visible(False)
            ax_.coords[1].set_ticklabel_visible(False)
            ax_.coords[0].set_axislabel("", visible=False)
            ax_.coords[1].set_axislabel("", visible=False)
        # ax3.coords[0].set_ticklabel_position("t")
        # ax3.coords[1].set_ticklabel_position("r")

        ax3.coords[0].set_axislabel(r"Helioprojective Longitude (Solar-X)", fontsize=11)
        ax3.coords[1].set_axislabel(r"Helioprojective Latitude (Solar-Y)", fontsize=11)

        for ax_ in (ax3,ax4):
            ax_.coords[0].set_ticks(spacing=40*u.arcsec)
            ax_.coords[1].set_ticks(spacing=40*u.arcsec)

        ax1.text(0.03,1 - 0.03, r"\textbf{a) H}$\boldsymbol{\beta}$",
                ha="left", va="top",
                transform=ax1.transAxes,
                fontsize=11,
                color="w",
                path_effects=[path_effects.withStroke(linewidth=2, foreground="black"),
                path_effects.Normal()])

        ax2.text(0.03,1 - 0.03, r"\textbf{b) H}$\beta$",
                ha="left", va="top",
                transform=ax2.transAxes,
                fontsize=11,
                color="w",
                path_effects=[path_effects.withStroke(linewidth=2, foreground="black"),
                path_effects.Normal()])

        ax3.text(0.03, 1 - 0.03*hri_pr_noproj_array.shape[2]/hri_pr_noproj_array.shape[1],
                r"\textbf{c) HRI\textsubscript{EUV}}",
                ha="left", va="top",
                transform=ax3.transAxes,
                fontsize=11,
                color="w",
                path_effects=[path_effects.withStroke(linewidth=2, foreground="black"),
                path_effects.Normal()])

        ax4.text(0.03, 1 - 0.03*hri_pr_noproj_array.shape[2]/hri_pr_noproj_array.shape[1],
                r"\textbf{d) HRI\textsubscript{EUV}}",
                ha="left", va="top",
                transform=ax4.transAxes,
                fontsize=11,
                color="w",
                path_effects=[path_effects.withStroke(linewidth=2, foreground="black"),
                path_effects.Normal()])

        text_vbi_time = ax2.text(0.99, 0.03,
                r"\textbf{" + \
                f'{Hbeta_date_obs[Hbeta_index_].strftime("%Y-%m-%dT%H:%M:%S")}' + r"}",
                ha="right", va="bottom", 
                transform=ax2.transAxes,
                fontsize=11,
                color="w",
                path_effects=[path_effects.withStroke(linewidth=2, foreground="black"),
                path_effects.Normal()])
        
        text_hri_time = ax4.text(0.99, 0.035,
                r"\textbf{" + \
                f'{hrieuv_date_ear[hrieuv_index_].strftime("%Y-%m-%dT%H:%M:%S")}' + r"}",
                ha="right", va="bottom", 
                transform=ax4.transAxes,
                fontsize=11,
                color="w",
                path_effects=[path_effects.withStroke(linewidth=2, foreground="black"),
                path_effects.Normal()])

        for ax_ in (ax1,ax2,ax3,ax4):
            ax_.set_axisbelow(False) 
            ax_.grid(True, color="w", ls=":", lw=0.6, alpha=0.6)

        

        # plt.savefig("../../figs//test_figs/vbi_hri_fieldline_movie_test.png", dpi=150)
        # plt.show()

        def update_fig(ii):
            hrieuv_index_ = ii
            Hbeta_index_ = np.argmin(np.abs(Hbeta_date_obs - hrieuv_date_ear[ii]))

            im1.set_data(Hbeta_pr_da[Hbeta_index_,:,:])
            im2.set_data(Hbeta_pr_da[Hbeta_index_,:,:])
            im3.set_data(hri_pr_noproj_array[ii,:,:])
            im4.set_data(hri_pr_noproj_array[ii,:,:])

            text_vbi_time.set_text(
                r"\textbf{" + \
                f'{Hbeta_date_obs[Hbeta_index_].strftime("%Y-%m-%dT%H:%M:%S")}' + r"}")
            text_hri_time.set_text(
                r"\textbf{" + \
                f'{hrieuv_date_ear[hrieuv_index_].strftime("%Y-%m-%dT%H:%M:%S")}' + r"}")

            return None
        
        anim = animation.FuncAnimation(fig, update_fig, frames=tqdm(range(len(hrieuv_date_ear))), # tqdm(range(0,50)), #
                                            blit=False)
        
        if save_dir is not None:
            anim.save(save_dir, fps=fps, dpi=dpi,writer='ffmpeg', 
                codec='libx264',
                # bitrate=6000,
                extra_args=['-pix_fmt', 'yuv420p',
                '-vf', 'crop=trunc(iw/2)*2:trunc(ih/2)*2'])

In [24]:
make_movie(save_dir="../../figs/ms_movie/vbi_hri_fieldline.mp4", fps=30, dpi=150)

  0%|          | 0/360 [00:00<?, ?it/s]